## **Support Vector Machine**

#### **Imports and Config**

In [ ]:
import json
import pandas as pd
import os
import mlflow
import mlflow.sklearn
import numpy as np
import optuna 
from optuna.integration.mlflow import MLflowCallback
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import optuna.visualization.matplotlib as optuna_mpl

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Run Configuration
MODEL_TYPE = "SVM"
MOMENT = "moment_1"

N_FOLDS = 5
RANDOM_STATE = 42
N_TRIALS = 25 # Optuna trials in tuned phase

TARGET_COL = "REBOUND_TYPE" # 1 = offensive, 0 = defensive
BALANCED   = True  # this file is the balanced version - see directory name
BASE_TAG   = "balanced" if BALANCED else "unbalanced"
SUFFIX     = f"_{BASE_TAG}"
DATA_PATH  = f"../../../../data/sequences/{MOMENT}/features/tabular/sequences_fe_tabular{SUFFIX}.csv"

GIT_COMMIT = "fd0a1ba"

# Overfitting control
# Composite objective: roc_auc_mean - ALPHA * (train_mean - test_mean)
# Higher ALPHA -> stronger regularization pressure in the search.
ALPHA = 0.1
# Hard cap: trials where gap > this value are scored as 0.0 and logged
# but effectively removed from contention. Set to 1.0 to disable.
MAX_OVERFIT_GAP = 0.15

# Permutation importance (final model)
N_REPEATS_PERM = 10  # repeats for permutation importance estimation

# MLflow
EXP_BASELINE = f"{MODEL_TYPE}_baseline"
EXP_TUNED = f"{MODEL_TYPE}_tuned"
EXP_FINAL = f"{MODEL_TYPE}_final"
MLFLOW_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', 'mlruns'))
RUN_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}"))

os.makedirs(MLFLOW_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(MLFLOW_DIR, ".trash"), exist_ok=True)
mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR.replace(os.sep, '/')}")

print(f"Artifacts -> {RUN_DIR}")
print(f"Overfitting - alpha={ALPHA}, hard cap={MAX_OVERFIT_GAP}")


#### **Data Loading & Feature Selection**

In [ ]:
df = pd.read_csv(DATA_PATH)

EXCLUDED_COLS = ["GAME_ID", "EVENT_NUM", "DETECTION_CONFIDENCE",
                  "SHOT_DETECTION_CONFIDENCE", "RIM_DETECTION_CONFIDENCE"]
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL and c not in EXCLUDED_COLS]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
groups = df["GAME_ID"].values

print(f"Dataset : {X.shape[0]} samples | {X.shape[1]} features")
print(f"Class balance - offensive: {y.mean():.3f}")

# Class-balance flag, logged alongside GIT_COMMIT
class_counts = pd.Series(y).value_counts()
CLASS_BALANCE_RATIO = class_counts.min() / class_counts.max()
IS_BALANCED = bool(np.isclose(CLASS_BALANCE_RATIO, 1.0, atol=0.02))

#### **Helpers**

In [ ]:
CV = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "f1": "f1_macro",
    "precision": "precision_macro",
    "recall": "recall_macro",
}

BASE_PARAMS = {
    "model_type": MODEL_TYPE,
    "moment": MOMENT,
    "git_commit": GIT_COMMIT,
    "n_features": len(FEATURE_COLS),
    "n_folds": N_FOLDS,
    "cv_strategy": "StratifiedGroupKFold",
    "random_state": RANDOM_STATE,
    "alpha": ALPHA,
    "max_overfit_gap": MAX_OVERFIT_GAP,
    "class_balanced": IS_BALANCED,
}


# Fit a MinMaxScaler PER FOLD (train only), then the model.
#
# Scaling happens inside CV so no validation-fold range leaks into training.
def wrap(model):
    return Pipeline([("scale", MinMaxScaler()), ("model", model)])


# Cross-validate and log metrics + overfitting diagnostics to MLflow.
#
# Returns a flat metrics dict that always includes:
#   - per-metric mean/std/train_mean for every SCORING key
#   - overfit_gap_<metric> = train_mean - test_mean
#   - composite_score = roc_auc_mean - alpha * overfit_gap_roc_auc
def run_cv_and_log(model, extra_params: dict, phase: str = None) -> dict:
    cv_results = cross_validate(
        wrap(model), X, y,
        groups=groups,
        cv=CV,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=-1,
    )

    metrics = {}
    for metric in SCORING:
        test_scores  = cv_results[f"test_{metric}"]
        train_scores = cv_results[f"train_{metric}"]
        test_mean = float(np.mean(test_scores))
        train_mean = float(np.mean(train_scores))
        gap = train_mean - test_mean

        metrics[f"{metric}_mean"] = test_mean
        metrics[f"{metric}_std"] = float(np.std(test_scores))
        metrics[f"{metric}_train_mean"] = train_mean
        metrics[f"overfit_gap_{metric}"] = gap

    metrics["composite_score"] = (
        metrics["roc_auc_mean"] - ALPHA * metrics["overfit_gap_roc_auc"]
    )

    mlflow.log_params({**BASE_PARAMS, **extra_params})
    mlflow.log_metrics(metrics)

    if phase in ("baseline", "final"):
        # Feature list/order the saved model was trained on (read by the holdout cell)
        features_path = os.path.join(RUN_DIR, "features.json")
        with open(features_path, "w") as f:
            json.dump(FEATURE_COLS, f, indent=2)

        cv_path = os.path.join(RUN_DIR, f"cv_results_{phase}.json")
        with open(cv_path, "w") as f:
            json.dump({k: v.tolist() for k, v in cv_results.items()}, f, indent=2)

    return metrics


def print_summary(label: str, metrics: dict) -> None:
    gap = metrics["overfit_gap_roc_auc"]
    print(
        f"{label}\n"
        f"  roc_auc   : {metrics['roc_auc_mean']:.4f} +/- {metrics['roc_auc_std']:.4f}  "
        f"(train {metrics['roc_auc_train_mean']:.4f}, gap {gap:+.4f})\n"
        f"  accuracy  : {metrics['accuracy_mean']:.4f} +/- {metrics['accuracy_std']:.4f}  "
        f"(train {metrics['accuracy_train_mean']:.4f}, gap {metrics['overfit_gap_accuracy']:+.4f})\n"
        f"  composite : {metrics['composite_score']:.4f}  "
        f"[= roc_auc - {ALPHA}*gap]"
    )

#### **Baseline**

In [ ]:
# SVM Baseline params
BASELINE_PARAMS = {
    "C": 1.0,
    "kernel": "rbf",
    "gamma": "scale",
    "class_weight": None,
}

mlflow.set_experiment(EXP_BASELINE)

with mlflow.start_run(run_name=MODEL_TYPE):
    model   = SVC(**BASELINE_PARAMS, random_state=RANDOM_STATE)
    metrics = run_cv_and_log(model, extra_params={**BASELINE_PARAMS, "phase": "baseline"}, phase="baseline")

print_summary("Baseline", metrics)

#### **Tuned (Optuna)**

**Objective**: `composite_score = roc_auc_mean - alpha * overfit_gap`  
Trials where `overfit_gap_roc_auc > MAX_OVERFIT_GAP` are penalized to 0.0.  

In [ ]:
mlflow.set_experiment(EXP_TUNED)

mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="composite_score",   # Optuna optimizes composite, not raw roc_auc
    create_experiment=False,
)

def progress_callback(study, trial):
    print(f"Trial {trial.number + 1}/{N_TRIALS} - composite: {trial.value:.4f} | best so far: {study.best_value:.4f}")


@mlflc.track_in_mlflow()
def objective(trial):
    kernel = trial.suggest_categorical("kernel", ["rbf", "sigmoid"])

    params = {
        "C" : trial.suggest_float("C", 1e-2, 1e2, log=True),
        "kernel": kernel,
    }

    # gamma only applies to rbf / sigmoid
    if kernel in ("rbf", "sigmoid"):
        params["gamma"] = trial.suggest_float("gamma", 1e-4, 1.0, log=True)

    model = SVC(**params, random_state=RANDOM_STATE)
    metrics = run_cv_and_log(model, extra_params={**params, "phase": "tuned"})

    gap = metrics["overfit_gap_roc_auc"]

    if gap > MAX_OVERFIT_GAP:
        mlflow.log_metric("pruned_overfit", 1.0)
        return 0.0

    mlflow.log_metric("pruned_overfit", 0.0)
    return metrics["composite_score"]


study = optuna.create_study(
    direction="maximize",
    study_name=f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, callbacks=[mlflc, progress_callback])

print(f"\nBest trial - composite_score: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Inspect overfitting
trials_df = study.trials_dataframe()

# Filter out pruned trials (composite = 0)
valid = trials_df[trials_df["value"] > 0].copy()

print(f"Valid trials (gap <= {MAX_OVERFIT_GAP}): {len(valid)} / {len(trials_df)}")
print(valid[["value"]].describe().T.to_string())

In [ ]:
with mlflow.start_run(run_name=f"best_{MODEL_TYPE}"):
    mlflow.log_params({
        **BASE_PARAMS,
        **study.best_params,
        "phase": "tuned_best",
        "source": "optuna_best",
    })
    mlflow.log_metric("composite_score", study.best_value)

    optuna_mpl.plot_param_importances(study)
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, "optuna_param_importance.png"), dpi=150)
    plt.close()

    optuna_mpl.plot_contour(study, params=["C", "gamma"])
    plt.tight_layout()
    plt.savefig(os.path.join(RUN_DIR, "optuna_contour_C_gamma.png"), dpi=150)
    plt.close()

#### **Final**

In [ ]:
FINAL_PARAMS = {**study.best_params}
# probability=True enables predict_proba for OOF capture below (Platt-scaling CV
# adds training cost to every fit() call, not just when predict_proba is used).
FINAL_PARAMS["probability"] = True

mlflow.set_experiment(EXP_FINAL)

with mlflow.start_run(run_name=MODEL_TYPE) as final_run:
    model = SVC(**FINAL_PARAMS, random_state=RANDOM_STATE)
    metrics = run_cv_and_log(model, extra_params={**FINAL_PARAMS, "phase": "final"}, phase="final")

    # Refit the full pipeline (scaler + model) on all data and log it as the artifact.
    # (Without the pipeline, model.fit(X, y) would fit an RBF SVM on UNSCALED data.)
    final_pipe = wrap(model)
    final_pipe.fit(X, y)
    mlflow.sklearn.log_model(final_pipe, "model")

FINAL_RUN_ID = final_run.info.run_id
print_summary("Final", metrics)

In [ ]:
from sklearn.model_selection import cross_val_predict

# Out-of-fold probabilities, used by calibrate.py
oof_probs = cross_val_predict(
    wrap(model), X, y,
    groups=groups,
    cv=CV,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]  # P(offensive)

oof_df = pd.DataFrame({
    "GAME_ID": df["GAME_ID"].values,
    "EVENT_NUM": df["EVENT_NUM"].values,
    "y_true": y,
    "y_prob": oof_probs,
})

oof_path = os.path.join(RUN_DIR, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(oof_path)

print(f"OOF predictions: {len(oof_df)} rows | true offensive rate {oof_df['y_true'].mean():.3f} | mean predicted prob {oof_df['y_prob'].mean():.3f}")

#### **Feature Importance (Final Model)**

SVMs have no native feature importances. Permutation importance is used instead:  
each feature is randomly shuffled in turn and the drop in `roc_auc` on a held-out  
validation fold is recorded as its importance.

In [ ]:
# Use the last CV fold's test split as the held-out set for permutation importance
train_idx, val_idx = list(CV.split(X, y, groups))[-1]
X_val, y_val = X[val_idx], y[val_idx]

perm = permutation_importance(
    final_pipe, X_val, y_val,
    n_repeats=N_REPEATS_PERM,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importances = pd.Series(perm.importances_mean, index=FEATURE_COLS)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot.barh(ax=ax)
ax.set_title("SVM Feature Importances - Top 20 (Permutation)")
ax.set_xlabel("Mean decrease in ROC-AUC")
plt.tight_layout()

fig_path = os.path.join(RUN_DIR, "feature_importances.png")
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved -> {fig_path}")

#### **Holdout Evaluation**

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

# Holdout counterpart of DATA_PATH (holdout is never balanced)
HOLDOUT_DATA_PATH = f"../../../../data/sequences/{MOMENT}/features/tabular/sequences_fe_tabular_holdout.csv"

df_holdout = pd.read_csv(HOLDOUT_DATA_PATH)

with open(os.path.join(RUN_DIR, "features.json")) as f:
    holdout_feature_cols = json.load(f)

missing = set(holdout_feature_cols) - set(df_holdout.columns)
assert not missing, f"Holdout is missing features used in training: {missing}"

X_holdout = df_holdout[holdout_feature_cols].values
y_holdout = df_holdout[TARGET_COL].values

y_pred = final_pipe.predict(X_holdout)
# SVC has no predict_proba unless probability=True was set; fall back to the
# decision function so roc_auc still works for every model in this template.
y_score = (final_pipe.predict_proba(X_holdout)[:, 1] if hasattr(final_pipe, "predict_proba")
           else final_pipe.decision_function(X_holdout))

holdout_metrics = {
    "accuracy": accuracy_score(y_holdout, y_pred),
    "roc_auc": roc_auc_score(y_holdout, y_score),
    "f1": f1_score(y_holdout, y_pred, average="macro"),
    "precision": precision_score(y_holdout, y_pred, average="macro"),
    "recall": recall_score(y_holdout, y_pred, average="macro"),
}

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_metrics({f"holdout_{k}": v for k, v in holdout_metrics.items()})
    mlflow.log_param("n_holdout", len(df_holdout))

print(f"Holdout: {len(df_holdout)} rows, offensive rate {y_holdout.mean():.3f}")
for k, v in holdout_metrics.items():
    print(f"  holdout_{k:9s}: {v:.4f}")